In [6]:
import pandas as pd
import ast
import re


# 1. File paths
BASE_PATH = "base_song_table_2008_2018_day_precision.csv"
ACOUSTIC_PATH = "musicoset_songfeatures/acoustic_features.csv"
LYRICS_PATH = "musicoset_songfeatures/lyrics.csv"
SONG_CHART_PATH = "musicoset_popularity/song_chart.csv"
OUTPUT_PATH = "base_song_table_2008_2018_enriched.csv"


# 2. Load base table
base = pd.read_csv(BASE_PATH)
base.columns = base.columns.str.strip()
base["song_id"] = base["song_id"].astype(str).str.strip()

print("Base shape:", base.shape)
print("Unique song_id in base:", base["song_id"].nunique())


# 3. Load acoustic features
acoustic = pd.read_csv(ACOUSTIC_PATH, sep="\t", low_memory=False)
acoustic.columns = acoustic.columns.str.strip()
acoustic["song_id"] = acoustic["song_id"].astype(str).str.strip()

print("\nAcoustic shape before dedup:", acoustic.shape)
acoustic = acoustic.drop_duplicates(subset=["song_id"])
print("Acoustic shape after dedup:", acoustic.shape)


# 4. Load and clean lyrics
def clean_lyrics_cell(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            x = " ".join(str(i) for i in parsed if pd.notna(i))
    except:
        pass

    # convert escaped sequences
    x = x.replace("\\n", "\n")
    x = x.replace("\\t", " ")
    x = x.replace("\\r", " ")

    # remove weird spaces
    x = x.replace("\xa0", " ")

    # collapse spaces but KEEP newlines
    x = re.sub(r"[ ]{2,}", " ", x)

    # normalize multiple blank lines
    x = re.sub(r"\n{3,}", "\n\n", x)

    return x.strip()


lyrics = pd.read_csv(LYRICS_PATH, sep="\t", low_memory=False)
lyrics.columns = lyrics.columns.str.strip()
lyrics["song_id"] = lyrics["song_id"].astype(str).str.strip()

print("\nLyrics shape before processing:", lyrics.shape)

# If there are multiple lyric rows per song, combine them into one text field
if lyrics.duplicated(subset=["song_id"]).any():
    lyrics = (
        lyrics.groupby("song_id", as_index=False)["lyrics"]
        .agg(lambda x: " ".join(x.dropna().astype(str).str.strip()))
    )
else:
    lyrics = lyrics.drop_duplicates(subset=["song_id"])

# Clean lyrics after grouping/deduplication
lyrics["lyrics"] = lyrics["lyrics"].apply(clean_lyrics_cell)

print("Lyrics shape after processing:", lyrics.shape)
print("Lyrics coverage after cleaning:", (1 - lyrics["lyrics"].isna().mean()))


# 5. Load song chart data
song_chart = pd.read_csv(SONG_CHART_PATH, sep="\t", low_memory=False)
song_chart.columns = song_chart.columns.str.strip()

print("\nSong chart original shape:", song_chart.shape)
print("Song chart original columns:", list(song_chart.columns))

# Rename id -> song_id so it matches the base table
song_chart = song_chart.rename(columns={"id": "song_id"})
song_chart["song_id"] = song_chart["song_id"].astype(str).str.strip()

# Aggregate chart variables to one row per song
chart_agg = (
    song_chart.groupby("song_id", as_index=False)
    .agg({
        "rank_score": "max",
        "peak_position": "min",
        "weeks_on_chart": "max"
    })
)

print("Chart aggregated shape:", chart_agg.shape)


# 6. Match rates BEFORE merge
acoustic_ids = set(acoustic["song_id"])
lyrics_ids = set(lyrics["song_id"])
chart_ids = set(chart_agg["song_id"])

print("Match rates BEFORE merge")
print(f"Acoustic match rate: {base['song_id'].isin(acoustic_ids).mean():.2%}")
print(f"Lyrics match rate:   {base['song_id'].isin(lyrics_ids).mean():.2%}")
print(f"Chart match rate:    {base['song_id'].isin(chart_ids).mean():.2%}")


# 7. Merge step by step
df = base.merge(acoustic, on="song_id", how="left")
print("\nAfter merging acoustic:", df.shape)

df = df.merge(lyrics, on="song_id", how="left")
print("After merging lyrics:", df.shape)

df = df.merge(chart_agg, on="song_id", how="left")
print("After merging chart:", df.shape)


# 8. Coverage AFTER merge
print("Coverage AFTER merge")

if "lyrics" in df.columns:
    print(f"Lyrics coverage:         {(1 - df['lyrics'].isna().mean()):.2%}")

if "rank_score" in df.columns:
    print(f"rank_score coverage:     {(1 - df['rank_score'].isna().mean()):.2%}")

if "peak_position" in df.columns:
    print(f"peak_position coverage:  {(1 - df['peak_position'].isna().mean()):.2%}")

if "weeks_on_chart" in df.columns:
    print(f"weeks_on_chart coverage: {(1 - df['weeks_on_chart'].isna().mean()):.2%}")

possible_acoustic_cols = [
    "acousticness", "danceability", "energy", "instrumentalness",
    "liveness", "loudness", "speechiness", "valence", "tempo",
    "duration_ms", "key", "mode", "time_signature"
]

for col in possible_acoustic_cols:
    if col in df.columns:
        print(f"{col} coverage:".ljust(26), f"{(1 - df[col].isna().mean()):.2%}")


# 9. Save output
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved to:", OUTPUT_PATH)
print("Final shape:", df.shape)

print("\nFinal columns:")
print(df.columns.tolist())

print("\nPreview:")
print(df.head())

Base shape: (6593, 8)
Unique song_id in base: 6593

Acoustic shape before dedup: (20405, 14)
Acoustic shape after dedup: (20405, 14)

Lyrics shape before processing: (20404, 2)
Lyrics shape after processing: (20404, 2)
Lyrics coverage after cleaning: 0.9636835914526564

Song chart original shape: (250392, 5)
Song chart original columns: ['song_id', 'rank_score', 'peak_position', 'weeks_on_chart', 'week']
Chart aggregated shape: (20402, 4)
Match rates BEFORE merge
Acoustic match rate: 100.00%
Lyrics match rate:   99.98%
Chart match rate:    99.97%

After merging acoustic: (6593, 21)
After merging lyrics: (6593, 22)
After merging chart: (6593, 25)
Coverage AFTER merge
Lyrics coverage:         95.65%
rank_score coverage:     99.97%
peak_position coverage:  99.97%
weeks_on_chart coverage: 99.97%
acousticness coverage:     100.00%
danceability coverage:     100.00%
energy coverage:           100.00%
instrumentalness coverage: 100.00%
liveness coverage:         100.00%
loudness coverage:    

In [7]:
print(df.loc[52])
print(df.loc[53])
print(df.loc[54])

song_id                                              73uJnm2f00Qlki0QSlEw9b
song_name                             Smoke! Smoke! Smoke! (That Cigarette)
artist_id                                            4KtaHyqLFlKKb7aghnWBfE
artist_name                       Commander Cody and His Lost Planet Airmen
album_id                                             4NyurQHqTZbhAA3FfC2f3y
album_name                                                              NaN
release_date                                                     2008-01-01
release_date_precision                                                  day
duration_ms                                                          191611
key                                                                       7
mode                                                                      1
time_signature                                                            4
acousticness                                                          0.807
danceability

In [8]:
print(type(df.loc[52, "lyrics"]))
print(repr(df.loc[52, "lyrics"]))

<class 'str'>
'Travis-Williams\n\nCho: Smoke! Smoke! Smoke that cigarette!\nPuff, Puff, Puff and if you smoke yourself to death\nTell Saint Peter at the Golden Gate\nThat you hate to make him wait\nBut you\'ve just got to have another cigarette\n\nNow I\'m a feller with a heart of gold\nAnd the ways of a gentleman, I\'ve been told\nThe kind of a guy that wouldn\'t even harm a flea--\nBut if me and a certain character met\nThe guy that invented the cigarette\nI\'d murder that son of a gun in the first degree--\nNot \'cause I don\'t smoke myself\nAnd I don\'t reckon they\'ll harm your health\nI\'ve smoked all my life and ain\'t dead yet\nBut nicotine slaves are all the same\nAt a pettin\' party or a poker game\nEverything must stop while they smoke that cigarette\n\nIn a game of chance the other night\nOld Dame Fortune was a-doin\' me right\nThe Kings and Queens just kept on comin\' around--\nI played \'em hard and bet \'em high\nBut my bluff didn\'t work on a certain guy\nHe kept on rai